# Tutorial: CrossConvBlock in PyTorch

This notebook demonstrates the use of the `CrossConvBlock` module from the
`neurite` library. The `CrossConvBlock` interacts a query image with a context set (of examples)
defining a task by performing crossconvolutions.

To learn more about this method, please check out:
- [UniverSeg Paper](https://arxiv.org/abs/2304.06131)
- [UniverSeg GitHub](https://github.com/JJGO/UniverSeg/tree/main?tab=readme-ov-file)

## Overview

The `CrossConvBlock` computes all combinations of a query image and a context set defining a task, 
and performs convolutions on the interactions 

1. **Expand:** Generate all combinations of query and context slices.
2. **Convolve:** Apply a standard convolution to the expanded tensor.
3. **Aggregate:** Average over the context set to obtain refined query features
   and vice versa.

In [1]:
import torch
from torch import nn
import einops

import neurite as ne
from neurite.pytorch.modules import CrossConvBlock

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## Creating Dummy Data

We create dummy 2D tensors for the query and context inputs. Here:

- **B:** Batch size
- **Sq:** Number of query images
- **Sc:** Number of members/image-seg pairs in the context set
- **Cq:** Number of channels in the query
- **Cc:** Number of channels in the combined context set
- **H, W:** Spatial dimensions

The query and context tensors are generated using random values.

In [9]:
# Define dummy data dimensions
B = 2            # Batch size
Sq = 1           # Number of query slices
Sc = 10          # Number of context slices
Cq = 1           # Channels in query
Cc = 2           # Channels in context
H, W = 64, 64    # Height and width

# Create random tensors for query and context
query = torch.randn(B, Sq, Cq, H, W, device=device)
context = torch.randn(B, Sc, Cc, H, W, device=device)

print('Query shape:', query.shape)
print('Context shape:', context.shape)

Query shape: torch.Size([2, 1, 1, 64, 64])
Context shape: torch.Size([2, 10, 2, 64, 64])


## Instantiating the CrossConvBlock

We create an instance of the **CrossConvBlock** for 2D convolutions. The block
requires a tuple for `in_channels` corresponding to the query and context, and
we set the output channels to 16.

In [13]:
# Instantiate the CrossConvBlock
cross_conv_block = CrossConvBlock(
    ndim=2,
    in_channels=(Cq, Cc),
    out_channels=16,
    kernel_size=3,
    padding=1,
    order="cacaca"
)

cross_conv_block.to(device)
print(cross_conv_block)

CrossConvBlock(
  (conv0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (query_conv_block): ConvBlock(
    (conv0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (context_conv_block): ConvBlock(
    (conv0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)


## Forward Pass

We pass the query and context tensors through the CrossConvBlock. The block
computes pairwise convolutions, aggregates the outputs by averaging over the
respective slices, and refines the features with additional convolutions.

In [15]:
# Perform the forward pass
new_query, new_context = cross_conv_block(query, context)

print('New query shape:', new_query.shape)
print('New context shape:', new_context.shape)

New query shape: torch.Size([2, 1, 16, 64, 64])
New context shape: torch.Size([2, 10, 16, 64, 64])


## Explanation of Outputs

- **New query shape:** The tensor shape is `(B, Sq, out_channels, H, W)`.
  This represents the refined query features after averaging/reducing over the
  context set.

- **New context shape:** The tensor shape is `(B, Sc, out_channels, H, W)`.
  This represents the refined context features after averaging/reducing over the
  query image(s).